# 국어나래 내장 코치 모델 파인튜닝 (Gemma 3 1B)

학생 기기 안에서 돌아가는 '생각 코치' 모델을 학습시킵니다. **학생에게 해가 되는 답을 하지 않는 것**이 최우선 목표입니다.

**실행 전 준비 3가지**
1. 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`(무료)
2. Hugging Face 토큰을 왼쪽 열쇠 아이콘(보안 비밀)에 `HF_TOKEN` 이름으로 넣고 '노트북 액세스' 켜기(맨 아래 7번 설명 참고)
3. 맨 아래 7번 칸의 `REPO`에 내 Hugging Face 아이디 넣기

준비가 끝나면 메뉴 `런타임 → 모두 실행`을 누르면 됩니다(대략 1~2시간, 브라우저 탭을 닫지 마세요).
**안전 시험을 통과한 모델이 없으면 5번에서 자동으로 멈추고**, 변환·업로드는 하지 않습니다.

**하는 일**
1. 학습 전 모델의 안전 시험(비교 기준)
2. SFT: 좋은 답(질문만)을 따라 하도록 학습
3. DPO: 같은 상황의 '안전한 답'과 '해로운 답'을 비교해 안전한 쪽을 고르도록 학습
4. 단계마다 **학습에 쓰지 않은** 레드팀 문항으로 안전 시험 → 모든 안전 기준을 통과한 모델만 채택
5. 앱이 읽는 형식(ONNX int8)으로 변환하고 Hugging Face에 올리기

**안전 설계(학습 데이터)**: 대필 요청(우회·압박·역할극), 학생 글 속 숨은 지시문, 모욕·폭력·욕설·차별·사생활 침해가 담긴 글, 위험 신호(괴롭힘·학대·자해), 위험 신호처럼 보이지만 아닌 표현(거짓 알림 방지), 개인 정보, 주제 이탈, 맞춤법 정답 요구.
앱의 가드레일·규칙 감지기는 그대로 유지됩니다(학습은 추가 안전망).

데이터 출처: Claude가 작성한 질문 은행·가상 학생 글을 프로그램으로 조합(실제 학생 데이터 없음). 교사가 약관 확인 후 사용 결정(2026-11-03).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 학습 코드와 데이터 받기(공개 저장소)
!git clone -q --depth 1 -b feat/module-core https://github.com/oswprogram10-tech/narae-kor-aidt.git
%cd narae-kor-aidt/training
!pip install -q -U transformers trl peft datasets accelerate

## 1. 학습 데이터 검수
선생님이 직접 몇 개를 읽어 보세요. 이상한 예가 있으면 학습 전에 알려 주세요(데이터 생성기: `training/gen/`).

In [ ]:
import json, random
print(open('data/stats.json', encoding='utf-8').read())
rows = [json.loads(l) for l in open('data/sft_train.jsonl', encoding='utf-8')]
random.seed(0)
for cat in ['help', 'ghostwrite', 'harmful_draft', 'distress', 'hard_negative', 'injection', 'personal', 'off_topic', 'spelling']:
    r = random.choice([x for x in rows if x['meta']['category'] == cat])
    print(f"\n[{cat}] {r['meta']['scenario']} / {r['meta']['stage']}")
    print('  학생:', r['messages'][1]['content'].split('학생 말:')[-1].split('\n')[0].strip())
    print('  목표 답:', r['messages'][2]['content'])

## 2. 학습 전 안전 시험(비교 기준)
학습하지 않은 Gemma 3 1B가 레드팀 문항(488개)에 어떻게 답하는지 봅니다. 시간을 아끼려면 `--limit 150`을 붙이세요.

In [ ]:
!python finetune.py eval --model unsloth/gemma-3-1b-it --out runs/base/redteam_outputs.jsonl --batch 16

## 3. SFT 학습(약 20~40분)

In [ ]:
!python finetune.py sft --base unsloth/gemma-3-1b-it --out runs/sft --epochs 2

In [ ]:
!python finetune.py eval --model runs/sft/merged --out runs/sft/redteam_outputs.jsonl --batch 16

## 4. DPO 학습(안전한 답 선호, 약 20~40분)

In [ ]:
!python finetune.py dpo --base runs/sft/merged --out runs/dpo

In [ ]:
!python finetune.py eval --model runs/dpo/merged --out runs/dpo/redteam_outputs.jsonl --batch 16

## 5. 채택할 모델 고르기(자동)
안전 시험에서 **모든 기준을 통과한 모델만** 고릅니다(DPO 우선, 그다음 SFT). 둘 다 불합격이면 여기서 멈춥니다. 그때는 위 시험 결과 표를 Claude에게 알려 주세요.

In [ ]:
import json, os
CHOSEN = None
for name in ['dpo', 'sft']:
    p = f'runs/{name}/redteam_outputs.jsonl.score.json'
    if os.path.exists(p):
        r = json.load(open(p, encoding='utf-8'))
        print(name, '→', '통과' if r['passed'] else '불합격')
        if r['passed'] and CHOSEN is None:
            CHOSEN = f'runs/{name}/merged'
if CHOSEN is None:
    raise RuntimeError('안전 시험을 통과한 모델이 없어 여기서 멈춥니다. 변환·업로드는 하지 않습니다. 위 결과를 Claude에게 알려 주세요.')
print('채택:', CHOSEN)

## 6. 앱 형식(ONNX int8)으로 변환(약 10~20분)

In [ ]:
!pip install -q optimum-onnx onnx onnxruntime
!python export_onnx.py --model {CHOSEN} --out export/narae-gemma-1b
!ls -la export/narae-gemma-1b export/narae-gemma-1b/onnx

## 7. Hugging Face에 올리기
1. https://huggingface.co 에 가입하고 Settings → Access Tokens에서 **Write** 토큰을 만드세요.
2. Colab 왼쪽 열쇠 아이콘(보안 비밀)에 이름 `HF_TOKEN`으로 토큰을 넣고 '노트북 액세스'를 켜세요.
3. 아래 `REPO`의 사용자 이름을 바꾸고 실행하세요. 학생 데이터는 들어 있지 않습니다. 공개(`PRIVATE=False`)로 두면 앱 설치 때 바로 받을 수 있고, 비공개로 하면 받을 때 토큰이 필요합니다.

In [ ]:
# 토큰: Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN을 넣고 '노트북 액세스'를 켜 두세요. 없으면 직접 입력 칸이 떠요.
import os, getpass
REPO = 'YOUR_HF_USERNAME/narae-coach-gemma3-1b'   # ← YOUR_HF_USERNAME 자리에 내 Username(프로필 주소의 아이디)
PRIVATE = False
assert not REPO.startswith('YOUR' + '_HF_'), 'REPO 첫 줄의 YOUR_HF_USERNAME 자리에 내 Username을 넣어 주세요(예: hong/narae-coach-gemma3-1b)'
assert REPO.count('/') == 1, 'REPO는 "Username/저장소이름" 모양이어야 해요'

token = os.environ.get('HF_TOKEN', '')
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN') or token
except Exception as e:
    print('Colab 보안 비밀에서 토큰을 못 읽었어요:', type(e).__name__, e)
if not token:
    token = getpass.getpass('Hugging Face 토큰(Write 권한)을 붙여 넣으세요: ')

# 앞 단계 설치로 바뀐 패키지와 섞이지 않게 새 프로세스에서 올립니다.
!rm -rf __pycache__ && git pull -q   # 올리기 스크립트(upload_hf.py) 최신본 받기(캐시 파일이 받기를 막지 않게 먼저 지움)
!pip install -q -U huggingface_hub
os.environ['HF_TOKEN'] = token.strip()
!python upload_hf.py --repo {REPO} {'--private' if PRIVATE else ''}


## 끝
저장소 주소(예: `내아이디/narae-coach-gemma3-1b`)를 Claude에게 알려 주세요. 앱과 같은 코드로 안전 시험을 다시 채점하고, 브라우저에서 속도·품질을 확인한 뒤 앱의 기본 내장 모델로 연결합니다.